# Task 5: Explore the Iris dataset

This lesson loads Iris, inspects its structure, checks missing values, and uses charts to understand each feature's distribution. You do not need to supply a CSV: Iris is bundled with scikit-learn.

Run the cells from top to bottom using the repository's `.venv` Python kernel. This task is **exploratory data analysis (EDA)**: learning what is in a dataset before deciding what to do with it.

## 1. Import tools and locate the project

A **library** contains reusable tools. pandas handles tables; NumPy supports numerical calculations; Matplotlib creates plots; scikit-learn provides the bundled Iris dataset.

The helper functions in `src/explore.py` draw the same charts for the script and notebook. The loading, inspection, missing-value check and summary commands are shown below so you can learn them directly.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from IPython.display import display

PROJECT_DIR = None
for parent in [Path.cwd(), *Path.cwd().parents]:
    for candidate in [parent, parent / "projects" / "05_iris_data_exploration"]:
        if (candidate / "src" / "explore.py").is_file():
            PROJECT_DIR = candidate
            break
    if PROJECT_DIR is not None:
        break
if PROJECT_DIR is None:
    raise FileNotFoundError("Open this notebook from inside the project repository.")
sys.path.insert(0, str(PROJECT_DIR / "src"))
from explore import (
    prepare_data, plot_histograms, plot_boxplots,
    plot_scatterplots, plot_pairplot, plot_correlations, main,
)
print("Project:", PROJECT_DIR)

## 2. Load Iris as a table

`load_iris(as_frame=True)` returns a container with measurements (`data`), numeric species codes (`target`), feature names and species names. A **DataFrame** is a table with rows and named columns.

Each row represents a flower sample. The four numeric measurements are **features**. Species is the categorical label, which could be a prediction target in a later modeling task. Here we use it to compare groups. Codes 0, 1 and 2 identify categories; their numeric distances have no biological meaning.

In [ ]:
iris = load_iris(as_frame=True)
FEATURES = list(iris.feature_names)
raw = iris.data.copy()
# Replace the integer codes with named categories for readable tables and plots.
raw["species"] = pd.Categorical.from_codes(iris.target.to_numpy(), iris.target_names)
print("Feature names:", FEATURES)
print("Species code mapping:", dict(enumerate(iris.target_names)))
display(raw.head())
display(raw.tail())

A **sepal** is an outer flower part; a **petal** is one of the flower's inner parts. All measurements here are in **centimeters (cm)**.

| Column | Meaning | Type |
| --- | --- | --- |
| sepal length (cm) | Length of the sepal | Numeric |
| sepal width (cm) | Width of the sepal | Numeric |
| petal length (cm) | Length of the petal | Numeric |
| petal width (cm) | Width of the petal | Numeric |
| species | setosa, versicolor or virginica | Categorical |

The numeric target code is not added as another measurement.

## 3. Inspect shape, types and class counts

`shape` returns `(number of rows, number of columns)`. `info()` prints column names, types and nonmissing counts. `dtypes` lists data types. A numeric dtype supports arithmetic; the categorical dtype represents named groups.

`value_counts()` counts how many rows belong to each species. Use it to check whether groups have similar sample sizes.

In [ ]:
print("Rows and columns:", raw.shape)
raw.info()
display(raw.dtypes.rename("Data type").to_frame())
species_counts = raw["species"].value_counts(sort=False)
display(species_counts.rename("Flower count").to_frame())

There are **150 rows and 5 columns**: four numeric features and one species column. Each species has **50 samples**, so the dataset is balanced by species. This describes the sample, not the proportions of those species in nature.

## 4. Check missing values and repeated rows

`isna()` marks missing cells as True. Summing counts them because True contributes 1. A missing measurement is different from a measured zero.

`duplicated()` counts repeated complete rows after the first occurrence. `duplicated(keep=False)` selects every row involved in a repeated group. Repeated rounded measurements can belong to different flowers, so this check does not by itself justify deleting observations.

In [ ]:
missing = pd.DataFrame({
    "missing_count": raw.isna().sum(),
    "missing_percent": raw.isna().mean() * 100,
})
display(missing)
print("Total missing cells:", int(raw.isna().sum().sum()))
print("Repeated rows beyond the first:", int(raw.duplicated().sum()))
display(raw.loc[raw.duplicated(keep=False)])

**Result:** There are **no missing values**, so no filling or dropping is necessary. Source row indices 101 and 142 have the same four measurements and virginica label. We retain both: there is no sample identifier or other evidence establishing a recording error.

If numeric measurements were missing, a median could be a simple starting choice after considering why values are missing. A column with no observed values has no usable median, and a missing species must not be guessed.

In [ ]:
# Work on a copy. This helper fills numeric gaps only when they exist.
# With the bundled dataset, it preserves every value and all 150 observations.
df, imputed = prepare_data(raw)
print("Feature cells filled:", int(imputed.sum()))
print("Rows after preparation:", len(df))
pd.testing.assert_frame_equal(df, raw)
print("Confirmed: no changes were necessary.")

The median branch is available in the script, but it is not used for this complete dataset. We do not manufacture missingness or alter measurements for the main analysis.

If you later train a model, split the data first and learn any filling values from training data only. The descriptive median policy here is not a fitted prediction pipeline.

## 5. Summarize distributions numerically

A **distribution** describes which values appear and how often. `describe()` gives:

- **count:** number of observed values;
- **mean:** arithmetic average;
- **std:** sample standard deviation, a measure of spread;
- **min / max:** smallest and largest observed values;
- **25%, 50%, 75%:** quartiles; 50% is the median.

`.T` transposes the table so features appear as rows.

In [ ]:
statistics = df[FEATURES].describe().T
display(statistics.round(3))
print("Mean measurements by species (cm):")
display(df.groupby("species", observed=True)[FEATURES].mean().round(3))

Across all flowers, sepal length ranges from **4.3 to 7.9 cm**, sepal width from **2.0 to 4.4 cm**, petal length from **1.0 to 6.9 cm**, and petal width from **0.1 to 2.5 cm**.

Averages alone hide groups. Comparing species means shows that setosa has the smallest average petals and virginica the largest. We now inspect the full distributions.

## 6. Histograms: how often do measurements occur?

A **histogram** groups numeric values into intervals called **bins**. The x-axis is the measured value in cm; the y-axis counts flowers in each interval.

The gray bars show the whole dataset. Colored outlines show each species using the **same bin edges**, so the distributions can be compared fairly. Changing the bin count can change the apparent shape.

In [ ]:
fig = plot_histograms(df)
display(fig)
plt.close(fig)

Read all four panels:

- **Sepal length:** measurements span a broad range, with substantial overlap among species.
- **Sepal width:** species overlap heavily; setosa tends toward wider sepals in this sample.
- **Petal length:** setosa forms a distinct short-petal group; versicolor and virginica have larger petals and some overlap.
- **Petal width:** setosa also forms a narrow, small-width group; virginica tends toward larger widths than versicolor.

The multiple groups in the pooled petal distributions reflect a mixture of species. A plot with several peaks is not automatically evidence of bad data.

## 7. Box plots: compare medians and spread

A **box plot** summarizes each feature separately for each species. The box covers the middle 50% of the data, from the first quartile (Q1) to the third (Q3). Its central line is the median. The **interquartile range (IQR)** is Q3 minus Q1.

The whiskers extend to the most extreme observed values within 1.5 IQR of the box. Points beyond the whiskers are possible outliers under this plotting rule. They are not automatically mistakes and we do not remove them.

In [ ]:
fig = plot_boxplots(df)
display(fig)
plt.close(fig)

The petal boxes show greater differences between species than the sepal boxes. Some panels contain points beyond the whiskers; these deserve attention if there is a substantive reason to suspect a measurement error. A rule that flags a point is not sufficient evidence to delete it.

## 8. Scatter plots: inspect two features together

A **scatter plot** places one sample at its x and y measurements. Colors and marker shapes identify species. Both axes are measurements in cm. Identical points may overlap, so the number of visible markers can be smaller than the row count.

In [ ]:
fig = plot_scatterplots(df)
display(fig)
plt.close(fig)

The sepal plot shows considerable mixing of species. The petal plot separates setosa clearly in this sample, while versicolor and virginica overlap. This suggests useful differences for later analysis; it does not measure the accuracy of a predictive model.

## 9. Pair plot: inspect all feature combinations

A **pair plot** arranges many small plots into a grid. On the diagonal, histograms show single-feature distributions. Off the diagonal, scatter plots show pairs of features. Read the column label as x and the row label as y. On the diagonal, y is a count.

This pair plot uses Matplotlib directly, so it does not require an additional plotting library.

In [ ]:
fig = plot_pairplot(df)
display(fig)
plt.close(fig)

## 10. Correlation: summarize linear relationships

**Pearson correlation** ranges from -1 to +1. Values near +1 indicate that larger measurements tend to occur together; values near -1 indicate an opposite trend. Values near 0 indicate weak **linear** association, which does not rule out other patterns.

Only the four numeric measurements belong in this table. Species codes are categories and should not be interpreted as numerical measurements.

In [ ]:
correlations = df[FEATURES].corr()
display(correlations.round(3))
fig = plot_correlations(df)
display(fig)
plt.close(fig)

Petal length and width have an overall correlation of about **0.963**, showing a strong positive association in this sample. Correlation does not establish that one measurement causes another. Combining species can strengthen or obscure relationships within a species.

If curious, compare the within-species correlations below with the pooled result:

In [ ]:
within_species = df.groupby("species", observed=True).apply(
    lambda group: group["petal length (cm)"].corr(group["petal width (cm)"])
)
display(within_species.rename("Petal length / width correlation").to_frame().round(3))

## 11. Save the analysis

The script saves the same dataset, quality checks, summary tables and five PNG figures. It also writes `reports/findings.md` with the results and explanations, and `summary.json` with software versions and a fingerprint of the exported CSV.

The original packaged dataset remains unchanged. Generated files follow the repository's existing Git ignore rules.

In [ ]:
main()
print("Findings:", PROJECT_DIR / "reports" / "findings.md")
print("Figures:", PROJECT_DIR / "reports" / "figures")

## 12. Conclusion and practice

**Example conclusion:** The Iris dataset contains 150 samples, four numeric measurements in centimeters, and three species with 50 samples each. No missing values were found, so no imputation was needed. One repeated row beyond its first occurrence was retained because equal measurements do not establish an error. Histograms and box plots show clearer species differences in petal measurements than sepal measurements. Scatter plots highlight setosa's distinct petal measurements, while versicolor and virginica overlap. Petal length and width are strongly positively correlated across all samples.

This small teaching sample does not establish population proportions or general model performance. We have explored the data without training a model.

**Check your understanding:**

1. Why does our DataFrame have five columns when Iris has four features?
2. Why did we leave the missing-value treatment unused?
3. What is the difference between a histogram and a scatter plot?
4. Does a point beyond a box plot's whiskers have to be removed?

**Answers:** 1. The fifth column is the species label. 2. There were no missing cells. 3. A histogram shows one feature's frequency distribution; a scatter plot shows two features together. 4. No: the rule flags a possible outlier, not a proven mistake.

## Sources

The analysis uses the dataset packaged with scikit-learn, rather than a separate CSV download. See the [official loader documentation](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html) and the saved `data/README.md` provenance record. The loader documents corrections relative to the original UCI copy.

Historical attribution: R. A. Fisher (1936), Iris dataset. The [UCI dataset record](https://www.archive.ics.uci.edu/dataset/53/iris) lists CC BY 4.0. Dependency versions and the local exported-data fingerprint are recorded in `reports/summary.json`.